# Differential Privacy Researcher Tutorial

This notebook is designed for DP researchers running RAG experiments.

What you will do:
- Build a DP corpus from a single `PrivacyConfig` (no epsilon mismatch).
- Run a multi-round agent and inspect an accumulated-RDP budget snapshot.
- Understand incremental privacy cost for the next retrieval round.
- Verify ergonomic behavior: budget is only debited after successful retrieval.

In [ ]:
from __future__ import annotations

import socket
import subprocess
import sys
import time
from types import SimpleNamespace

import httpx

from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.backend_client import create_backend
from securerag.budget import BudgetManager
from securerag.corpus import CorpusBuilder
from securerag.errors import BackendError
from securerag.models import RawDocument
from securerag.retriever import PrivacyRetriever
from securerag.llm import OllamaLLM

import securerag.retrievers  # ensure protocol retrievers are registered


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def wait_for_health(base_url: str, timeout: float = 10.0) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = httpx.get(f"{base_url}/health", timeout=0.4)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")


def print_snapshot(snap: dict) -> None:
    print("spent:", round(snap["spent"], 6))
    print("remaining:", round(snap["remaining"], 6))
    print("rounds:", snap["rounds"])
    print("ledger:", [(r, round(eps, 6)) for r, eps in snap["ledger"]])

## 1) Budget Intuition: Accumulated RDP vs Naive Per-Round Sum

The current budget manager composes in RDP and converts to $(\epsilon, \delta)$ after accumulation.

This cell prints how epsilon grows across rounds for several noise levels.

In [ ]:
for sigma in [0.8, 1.0, 1.5]:
    cfg = PrivacyConfig(protocol=PrivacyProtocol.DIFF_PRIVACY, epsilon=100.0, delta=1e-5, noise_std=sigma)
    b = BudgetManager(cfg)
    row = []
    for _ in range(4):
        b.consume(sigma=sigma)
        row.append(round(b.spent, 4))
    print(f"sigma={sigma}: epsilon trajectory by round -> {row}")

## 2) Build a DP Corpus from PrivacyConfig

`CorpusBuilder.from_config(...)` wires server-side epsilon/delta from the same config used by the agent.

In [ ]:
port = free_port()
base_url = f"http://127.0.0.1:{port}"
sim_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "securerag.sim_server:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_health(base_url)

docs = [
    RawDocument(doc_id="q3", text="Q3 risk report highlights vendor concentration and delayed remediation."),
    RawDocument(doc_id="policy", text="Policy requires quarterly risk treatment tracking and ownership."),
    RawDocument(doc_id="ops", text="Operational incidents increased due to queue saturation in ingestion."),
]

cfg = PrivacyConfig(
    protocol=PrivacyProtocol.DIFF_PRIVACY,
    epsilon=9.0,
    delta=1e-5,
    noise_std=1.0,
    top_k=2,
    max_rounds=4,
    backend=base_url,
)

corpus = (
    CorpusBuilder.from_config(cfg)
    .with_chunk_size(180)
    .add_documents(docs)
    .build_local(workers=2)
)

print("Corpus protocol:", corpus.protocol)
print("Index id:", corpus.index_id)
print("Doc count:", corpus.meta.doc_count)

## 3) Run a Multi-Round DP Agent and Inspect Budget

Use `OllamaLLM(use_ollama=False)` for deterministic local fallback behavior in tutorials.

In [ ]:
agent = SecureRAGAgent.from_config(
    cfg,
    corpus=corpus,
    llm=OllamaLLM(use_ollama=False),
)

result = agent.run("What are the key Q3 risk concerns?")
snapshot = agent.budget_snapshot()

print("Answer preview:", result.answer[:140])
print("Context size:", result.context_size)
print("Rounds used:", result.rounds)
print_snapshot(snapshot)

next_cost = agent.retriever.privacy_cost("another follow-up query")
print("Projected incremental epsilon for one more round:", round(next_cost, 6))

## 4) Ergonomics Check: Failed Retrieval Does Not Burn Budget

This demonstrates success-only debit behavior in `DiffPrivacyRetriever`.

In [ ]:
class FailingBackend:
    def embed_with_noise(self, query: str, sigma: float) -> list[float]:
        return [0.0] * 64

    def retrieve_by_embedding(self, **kwargs):
        raise BackendError("simulated backend failure")


local_cfg = PrivacyConfig(
    protocol=PrivacyProtocol.DIFF_PRIVACY,
    epsilon=5.0,
    delta=1e-5,
    noise_std=1.0,
    backend="memory://unused",
)
local_corpus = SimpleNamespace(protocol=PrivacyProtocol.DIFF_PRIVACY, index_id="idx")
local_retriever = PrivacyRetriever.from_config(local_cfg, local_corpus)
local_retriever._backend = FailingBackend()

try:
    local_retriever.retrieve("q3 risk", round_n=0)
except BackendError as exc:
    print("Caught expected backend error:", exc)

print_snapshot(local_retriever.budget.snapshot())

In [ ]:
# Cleanup local server process started in this tutorial.
sim_proc.terminate()
try:
    sim_proc.wait(timeout=2)
except subprocess.TimeoutExpired:
    sim_proc.kill()

print("sim_server stopped")